# Azure Observable RAG — Evaluation Harness

Two layers of metrics, both consuming the same `FinalRagTrace`:

**Retrieval (programmatic, against gold chunk_ids)**
- `recall@k`
- `MRR`

**Generation**
- `citation_correctness` — programmatic: every cited chunk_id is actually in the context
- `groundedness` — LLM-as-judge (gpt-4o-mini, JSON mode), 1–5
- `answer_relevance` — LLM-as-judge, 1–5

**Targets:** recall@5 ≥ 0.7, groundedness ≥ 4/5, citation_correctness = 1.0

**Setup:** edit `notebooks/gold_qa.json` to fill in the `gold_chunk_ids` for each question. To get a chunk_id, run `python -m src.cli search "<question>" --json | jq -r '.[].chunk_id'` and pick the relevant ones.

In [ ]:
import sys, json
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

import pandas as pd
from src.eval_harness import run_eval

## Run the harness

Each question is sent through the full LangGraph; results stream to `eval_results.jsonl`.

In [ ]:
out = run_eval(gold_path=str(REPO_ROOT / 'notebooks' / 'gold_qa.json'))
summary = out['summary']
rows = out['rows']
summary

## Per-question results

In [ ]:
df = pd.DataFrame(rows)
df[['question', 'intent', 'recall_at_k', 'mrr', 'citation_correctness', 'groundedness', 'answer_relevance']]

## Aggregate against targets

In [ ]:
targets = {
    'recall_at_k_mean':         (summary['recall_at_k_mean'], 0.70),
    'mrr_mean':                  (summary['mrr_mean'], 0.50),
    'citation_correctness_rate': (summary['citation_correctness_rate'], 1.00),
    'groundedness_mean':         (summary['groundedness_mean'], 4.00),
    'answer_relevance_mean':     (summary['answer_relevance_mean'], 4.00),
}
rep = pd.DataFrame(
    [{'metric': k, 'observed': v[0], 'target': v[1], 'pass': (v[0] is not None and v[0] >= v[1])} for k, v in targets.items()]
)
rep

## Inspect failure cases

In [ ]:
fails = df[(df['groundedness'] < 4) | (df['answer_relevance'] < 4) | (~df['citation_correctness'])]
fails[['question', 'groundedness', 'groundedness_reason', 'answer_relevance', 'answer_relevance_reason', 'citation_correctness']]